# Project BRAHMA — SadTalker Hindi Avatar Generation

Reproducible Google Colab workflow for generating a talking-avatar MP4 from a source photo and Hindi voice recording.

## Verified workflow design
- Colab T4 GPU; Miniconda3 with Python 3.8.20
- PyTorch 2.0.1, torchvision 0.15.2, torchaudio 2.0.2, CUDA 11.8
- OpenTalker/SadTalker source and requirements
- Four primary SadTalker checkpoints
- Google Drive inputs: `BRAHMA_SadTalker/inputs/brahma_photo.jpeg` and `brahma_voice.mp3`
- Convert audio to 16 kHz mono WAV
- Run inference with 512 resolution, crop preprocessing, pose_style 0, batch_size 2
- Save output to temporary Colab local storage and download directly to the computer

**Privacy/storage:** This notebook does not upload personal media, model weights, or generated videos to GitHub. It does not save generated MP4s to Drive, avoiding Drive quota issues. Colab `/content` is temporary; download the final MP4 before ending the runtime.

Select Runtime → Change runtime type → T4 GPU, then run cells in order. If a cell errors, stop and inspect the full output before continuing.

## 1. Check GPU runtime

In [ ]:
import sys, subprocess
print("Python:", sys.version)
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr)
if r.returncode != 0:
    raise RuntimeError("GPU unavailable. Select a T4 GPU runtime.")

## 2. Mount Drive and verify source inputs

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/BRAHMA_SadTalker")
INPUT_DIR = DRIVE_ROOT / "inputs"
PHOTO_PATH = INPUT_DIR / "brahma_photo.jpeg"
AUDIO_MP3_PATH = INPUT_DIR / "brahma_voice.mp3"

LOCAL_ROOT = Path("/content/BRAHMA_SadTalker")
LOCAL_OUTPUT_DIR = LOCAL_ROOT / "outputs"
LOCAL_CHECKPOINT_DIR = LOCAL_ROOT / "checkpoints"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Photo:", PHOTO_PATH.exists(), PHOTO_PATH)
print("Audio:", AUDIO_MP3_PATH.exists(), AUDIO_MP3_PATH)
if not PHOTO_PATH.exists() or not AUDIO_MP3_PATH.exists():
    raise FileNotFoundError("Expected photo/audio files are missing from Drive inputs.")

## 3. Install Miniconda3 and create Python 3.8.20 environment

In [ ]:
import subprocess
from pathlib import Path

MINICONDA_DIR = Path("/content/miniconda3")
CONDA = MINICONDA_DIR / "bin/conda"
if not CONDA.exists():
    installer = "/content/miniconda.sh"
    subprocess.run(["bash", "-lc",
        f"wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O {installer} && "
        f"bash {installer} -b -p {MINICONDA_DIR}"], check=True)

if not (MINICONDA_DIR / "envs/sadtalker/bin/python").exists():
    subprocess.run([str(CONDA), "create", "-y", "-n", "sadtalker", "python=3.8.20"], check=True)

PYTHON = str(MINICONDA_DIR / "envs/sadtalker/bin/python")
subprocess.run([PYTHON, "--version"], check=True)

## 4. Install PyTorch 2.0.1 with CUDA 11.8

In [ ]:
subprocess.run([
    str(CONDA), "install", "-y", "-n", "sadtalker",
    "pytorch=2.0.1", "torchvision=0.15.2", "torchaudio=2.0.2",
    "pytorch-cuda=11.8", "-c", "pytorch", "-c", "nvidia"
], check=True)

## 5. Install system tools

In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "git", "wget"], check=True)
print("ffmpeg, git and wget are ready.")

## 6. Clone OpenTalker/SadTalker

In [ ]:
REPO = Path("/content/SadTalker")
if not (REPO / "inference.py").exists():
    if REPO.exists():
        subprocess.run(["rm", "-rf", str(REPO)], check=True)
    subprocess.run(["git", "clone", "https://github.com/OpenTalker/SadTalker.git", str(REPO)], check=True)
assert (REPO / "inference.py").exists()
print("Repository ready:", REPO)

## 7. Install SadTalker requirements

In [ ]:
subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
print("Requirements installation completed.")

## 8. Verify framework versions and CUDA

In [ ]:
verify = """
import torch, torchvision, numpy, cv2
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('NumPy:', numpy.__version__)
print('OpenCV:', cv2.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
"""
r = subprocess.run([PYTHON, "-c", verify], capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print(r.stderr[-2500:])
if r.returncode != 0 or "CUDA available: True" not in r.stdout:
    raise RuntimeError("Dependency/CUDA verification failed.")

## 9. Restore or download the four main model checkpoints

In [ ]:
import shutil

DRIVE_CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_urls = {
    "mapping_00109-model.pth.tar": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar",
    "mapping_00229-model.pth.tar": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar",
    "SadTalker_V0.0.2_256.safetensors": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors",
    "SadTalker_V0.0.2_512.safetensors": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors",
}
for name, url in checkpoint_urls.items():
    local_file = LOCAL_CHECKPOINT_DIR / name
    drive_file = DRIVE_CHECKPOINT_DIR / name
    if local_file.exists() and local_file.stat().st_size > 1_000_000:
        print("Already local:", name)
    elif drive_file.exists() and drive_file.stat().st_size > 1_000_000:
        shutil.copy2(drive_file, local_file)
        print("Copied from Drive:", name)
    else:
        print("Downloading to Colab local storage:", name)
        subprocess.run(["wget", "-c", "--show-progress", url, "-O", str(local_file)], check=True)
    if not local_file.exists() or local_file.stat().st_size < 1_000_000:
        raise RuntimeError("Checkpoint missing or incomplete: " + name)
print("All four checkpoints are present locally.")
for p in sorted(LOCAL_CHECKPOINT_DIR.iterdir()):
    print(p.name, f"{p.stat().st_size / (1024**2):.1f} MB")

## 10. Connect SadTalker to local checkpoint directory

In [ ]:
REPO_CHECKPOINT_DIR = REPO / "checkpoints"
if REPO_CHECKPOINT_DIR.is_symlink():
    REPO_CHECKPOINT_DIR.unlink()
elif REPO_CHECKPOINT_DIR.exists():
    shutil.rmtree(REPO_CHECKPOINT_DIR)
REPO_CHECKPOINT_DIR.symlink_to(LOCAL_CHECKPOINT_DIR, target_is_directory=True)

for name in ["SadTalker_V0.0.2_512.safetensors", "mapping_00229-model.pth.tar"]:
    p = REPO_CHECKPOINT_DIR / name
    print(("✓" if p.exists() else "✗"), name)
    if not p.exists():
        raise FileNotFoundError(p)
print("Checkpoint connection successful.")

## 11. Convert Hindi MP3 to 16 kHz mono WAV

In [ ]:
WAV_PATH = Path("/content/brahma_voice.wav")
subprocess.run([
    "ffmpeg", "-y", "-i", str(AUDIO_MP3_PATH),
    "-ar", "16000", "-ac", "1", str(WAV_PATH)
], check=True)
print("WAV size:", f"{WAV_PATH.stat().st_size / 1024:.1f} KB")
subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(WAV_PATH)
], check=True)

## 12. Final input/model checks and inference CLI check

In [ ]:
assert (REPO / "inference.py").exists()
assert (REPO_CHECKPOINT_DIR / "SadTalker_V0.0.2_512.safetensors").exists()
assert (REPO_CHECKPOINT_DIR / "mapping_00229-model.pth.tar").exists()
assert (REPO / "src/config/similarity_Lm3D_all.mat").exists()
assert PHOTO_PATH.exists() and WAV_PATH.exists()
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

r = subprocess.run([PYTHON, "inference.py", "--help"], cwd=str(REPO), capture_output=True, text=True)
print(r.stdout[:4500])
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("SadTalker inference CLI check failed.")
print("All required files are ready.")

## 13. Generate talking-avatar video (512 resolution)

In [ ]:
command = [
    PYTHON, "inference.py",
    "--driven_audio", str(WAV_PATH),
    "--source_image", str(PHOTO_PATH),
    "--checkpoint_dir", str(REPO_CHECKPOINT_DIR),
    "--result_dir", str(LOCAL_OUTPUT_DIR),
    "--size", "512",
    "--preprocess", "crop",
    "--pose_style", "0",
    "--batch_size", "2",
]
result = subprocess.run(command, cwd=str(REPO), text=True)
print("Inference return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("SadTalker inference failed.")

## 14. Locate generated MP4

In [ ]:
videos = sorted(LOCAL_OUTPUT_DIR.rglob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if not videos:
    raise FileNotFoundError("No MP4 found in local output folder.")
for video in videos:
    print(video.name, f"{video.stat().st_size / (1024*1024):.2f} MB", video)
VIDEO_PATH = videos[0]
print("Selected video:", VIDEO_PATH)

## 15. Preview the generated video

In [ ]:
from IPython.display import Video, display
display(Video(str(VIDEO_PATH), embed=True, width=640))

## 16. Download MP4 directly to your computer

In [ ]:
from google.colab import files
files.download(str(VIDEO_PATH))

## Operational notes
- Colab `/content` is temporary. Download the MP4 before ending the session.
- The notebook reads personal inputs from Drive but never uploads personal photos, voice recordings, generated videos, credentials, or model binaries to GitHub.
- Checkpoints are restored from Drive when present; missing checkpoints download to local Colab storage.
- Generated MP4s are saved locally, not to Drive. This avoids Drive storage quota upload failures.
- The optional BFM_Fitting archive and GFPGAN enhancer weights are not included in this basic verified inference path.
- SadTalker source and model licenses/terms should be reviewed before redistribution or commercial use.
- The exact dependency stack was verified in the earlier working Colab session; re-run all cells from a fresh runtime to validate reproducibility after future upstream changes.